# 3-Phase Class 0.5S Energy Meter Simulation
**Architecture:** TI ADS131M08 (24-bit Δ-Σ) + TIDA-010243 Reference Design  
**Standard:** IEC 62053-22 Class 0.5S  
**Grid:** 3-phase 4-wire Wye, 230 V L-N, 50 Hz, India  
**Repo:** [github.com/muthamizhselvan152/energy-meter-simulation](https://github.com/muthamizhselvan152/energy-meter-simulation)

---
Run cells **sequentially top to bottom**. All parameters live in `m00_config.py`.

| Module | Description |
|--------|-------------|
| M0 | System configuration & constants |
| M1 | Signal generation + 24-bit ADC quantization |
| M2 | True RMS — Vrms, Irms, VLL, unbalance |
| M3 | Active / Reactive / Apparent power & PF |
| M4 | FFT harmonic analysis + THD |
| M5 | Energy accumulation — kWh, kVARh, kVAh |
| M6 | IEC 62053-22 Class 0.5S accuracy verification |
| M7 | Full system dashboard (12 panels) |
| M8 | Modbus RTU/TCP register map |

In [ ]:
# Install / verify dependencies (Colab already has these)
import subprocess
subprocess.run(["pip", "install", "numpy", "matplotlib", "scipy", "-q"])
print("Dependencies ready")

In [ ]:
# Clone the repo (first time only)
import os
if not os.path.exists("energy-meter-simulation"):
    !git clone https://github.com/muthamizhselvan152/energy-meter-simulation.git
    print("Repo cloned")
else:
    !git -C energy-meter-simulation pull
    print("Repo updated")

import sys
sys.path.insert(0, "energy-meter-simulation/modules")
print("Module path set")

## Module 0 — System Configuration

In [ ]:
from m00_config import *
print_config()

## Module 1 — Signal Generation + 24-bit Quantization

In [ ]:
from m01_signal_gen import generate_signals, plot_signals

t, v_actual, i_actual, v_counts, i_counts, v_recon, i_recon = generate_signals()
plot_signals(t, v_actual, i_actual, v_counts, i_counts, v_recon, i_recon)

---
## Module 2 — True RMS Computation
Computes per-phase Vrms and Irms using the true RMS definition:
`Vrms = sqrt(mean(v²))` over the full 1600-sample window.  
Also computes line-to-line voltages (V_AB, V_BC, V_CA) and
NEMA MG1 voltage/current unbalance percentages.

In [ ]:
from m02_rms import compute_rms, plot_rms

results_rms = compute_rms(v_recon, i_recon)
plot_rms(results_rms)

---
## Module 3 — Active / Reactive / Apparent Power & Power Factor
Computes true power quantities for each phase and 3-phase totals:
- `P = mean(v · i)` — active power (W), valid for non-sinusoidal waveforms
- `S = Vrms × Irms` — apparent power (VA)
- `Q = sqrt(S² − P²)` — reactive power (VAR)
- `PF = P / S` — true power factor (includes harmonic distortion)

In [ ]:
from m03_power import compute_power, plot_power

results_power = compute_power(v_recon, i_recon)
plot_power(results_power)

---
## Module 4 — FFT Harmonic Analysis + THD
Applies a Hanning-windowed FFT to extract the harmonic spectrum:
- One-sided amplitude spectrum (V and A) for all 3 phases
- Per-harmonic magnitudes at orders 1–31 (50 Hz – 1550 Hz)
- `THD = sqrt(sum(Vh² for h=2..31)) / V1 × 100` per phase

Frequency resolution = 5 Hz/bin — harmonics land exactly on bin centres.

In [ ]:
from m04_harmonics import compute_harmonics, plot_harmonics

results_harmonics = compute_harmonics(v_recon, i_recon)
plot_harmonics(results_harmonics)

---
## Module 5 — Energy Accumulation (kWh, kVARh, kVAh)
Scales the 200 ms power window to energy units:  
`kWh = P_total × T_hours` where `T_hours = 1600/8000/3600 ≈ 5.556×10⁻⁵ h`

Also projects to hourly and daily consumption rates.

In [ ]:
from m05_energy import compute_energy, plot_energy

results_energy = compute_energy(results_power)
plot_energy(results_energy)

---
## Module 6 — IEC 62053-22 Class 0.5S Accuracy Verification
Runs 6 standardised test points with sinusoidal inputs (IEC 62053-22 Clause 8.2):

| # | Test condition | Error limit |
|---|----------------|-------------|
| 1 | 100% Ib, PF=1.0 | ±0.5% |
| 2 | 100% Ib, PF=0.5 lag | ±0.5% |
| 3 | 100% Ib, PF=0.8 lag | ±0.5% |
| 4 | 20% Ib, PF=1.0 | ±0.5% |
| 5 | 5% Ib, PF=1.0 | ±1.0% |
| 6 | 1% Ib, PF=1.0 | ±1.5% |

`Error% = (P_measured − P_reference) / P_reference × 100`

In [ ]:
from m06_accuracy import verify_accuracy, plot_accuracy

results_accuracy = verify_accuracy(v_recon, i_recon)
plot_accuracy(results_accuracy)

---
## Module 7 — Full System Dashboard
12-panel overview combining all module outputs into a single figure:

| Row | Panels |
|-----|--------|
| 0 | Voltage waveforms · Current waveforms · Vrms/Irms % · Power factor |
| 1 | Per-phase P/Q/S · 3-phase totals · THD · Voltage spectrum |
| 2 | IEC accuracy errors · Energy rates · Unbalance · System summary |

In [ ]:
from m07_dashboard import plot_dashboard

plot_dashboard(
    t, v_recon, i_recon,
    results_rms, results_power,
    results_harmonics, results_energy,
    results_accuracy
)

---
## Module 8 — Modbus RTU/TCP Register Map
Maps all computed values to 16-bit Modbus holding registers (FC03).  
Follows energy-meter conventions (Eastron SDM630 / ISKRA MT174 style):
- Floats stored as `uint16` with a fixed scale factor (master divides to recover SI units)
- Energy registers use 32-bit hi/lo word pairs (scale ×10000, resolution 0.0001 kWh)
- Simulates a complete RTU frame with CRC-16 checksum (polynomial 0xA001)

In [ ]:
from m08_modbus import print_modbus_map

print_modbus_map(
    results_rms, results_power,
    results_harmonics, results_energy,
    results_accuracy
)